In [2]:
import numpy as np
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from transformers import DataCollatorWithPadding
from scipy.special import softmax
import evaluate

In [3]:
dataset = load_dataset("stanfordnlp/imdb")

model_name = "distilbert/distilbert-base-uncased"

id2label = {0: "neg", 1:"pos"}
label2id = {"neg" : 0, "pos" : 1}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, id2label=id2label, label2id=label2id)

def pre_processing(x):
    return tokenizer(x["text"], truncation=True)

tokenized_data = dataset.map(pre_processing, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
accuracy = evaluate.load("accuracy")
auc_score = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    probabilities = softmax(predictions, axis=-1)

    positive_class_probs = probabilities[:, 1]

    auc_result = auc_score.compute(prediction_scores=positive_class_probs, references=labels)

    auc = np.round(auc_result['roc_auc'], 3) if (auc_result and 'roc_auc' in auc_result) else 0.0

    predicted_classes = np.argmax(predictions, axis=1)

    acc_result = accuracy.compute(predictions=predicted_classes, references=labels)

    acc = np.round(acc_result['accuracy'], 3) if (acc_result and 'accuracy' in acc_result) else 0.0

    return {"Accuracy": acc, "AUC": auc}

In [5]:
lr = 2e-5
batch_size = 16
num_epochs = 3

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate = lr,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy='epoch',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True
)

In [6]:
trainer = Trainer(
    model = model, 
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,0.266039,0.207685,0.921000,0.978000
2,0.153482,0.240825,0.930000,0.980000
3,0.087530,0.284862,0.932000,0.980000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4689, training_loss=0.16901715799902364, metrics={'train_runtime': 1232.3364, 'train_samples_per_second': 60.86, 'train_steps_per_second': 3.805, 'total_flos': 9834539051060448.0, 'train_loss': 0.16901715799902364, 'epoch': 3.0})